# ⚡ Parallele Tools & Sub-Agents — der Agent delegiert

In den anderen Notebooks lief der Agent **streng seriell**: ein Tool nach dem anderen.
Das ist oft Verschwendung. Zwei Muster machen Agenten schneller *und* mächtiger:

| Muster | Idee | Wann |
|---|---|---|
| **Parallele Tool-Calls** | Das Modell gibt **mehrere** `tool_calls` in *einer* Antwort zurück. Wir führen sie **nebenläufig** aus statt nacheinander. | Unabhängige Lookups (3 Städte, 5 Dateien, 4 APIs) |
| **Sub-Agents als Tool** | Ein **Orchestrator** hat ein Tool `delegate(auftrag)`. Der Aufruf startet einen **eigenen Agent-Loop** mit eigenem Kontext + eigenen Tools und liefert nur das **Ergebnis** zurück. | Aufgabe in unabhängige Teilaufgaben zerlegbar (Supervisor / Map-Reduce) |

Beide Muster kombinieren sich: Der Orchestrator delegiert **mehrere** Sub-Agents *parallel*, und jeder Sub-Agent ruft seine Tools wiederum *parallel* auf.

```
  Orchestrator
    └─ delegate("Steckbrief Wien")   ┐
    └─ delegate("Steckbrief Berlin") ├─ laufen GLEICHZEITIG (parallele Tool-Calls)
    └─ delegate("Steckbrief Tokio")  ┘
          └─ jeder Sub-Agent ruft wetter/einwohner/zeitzone  ← auch parallel
    → Orchestrator fasst die 3 Ergebnisse zusammen
```

> Es bleibt **derselbe Agent-Loop**. Neu ist nur: (a) wie wir die Tool-Batch *ausführen* und (b) dass ein Tool selbst wieder ein Agent sein darf.

## 0 · Setup

`llm()` wie gewohnt (Azure OpenAI aus der `.env`). Dazu `ThreadPoolExecutor` für die Nebenläufigkeit
und ein `vprint` mit Lock, damit Ausgaben aus parallelen Threads sich nicht zerreißen.

In [1]:
import os, json, time, threading
from concurrent.futures import ThreadPoolExecutor
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

def llm(messages, tools=None, tool_choice="auto"):
    """Unser einziger Draht zum Modell - identisch zu den anderen Notebooks.
    Der OpenAI/Azure-Client ist thread-safe, daher dürfen mehrere Sub-Agents
    parallel hier hineinrufen."""
    kwargs = dict(model=DEPLOYMENT, messages=messages)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

_print_lock = threading.Lock()
def vprint(*a):
    """Thread-sicheres print - sonst überlappen die Zeilen paralleler Agenten."""
    with _print_lock:
        print(*a)

print("Setup bereit. Deployment:", DEPLOYMENT)

Setup bereit. Deployment: gpt-5.4-mini


## 1 · Drei „langsame" Tools (simulierte Latenz)

Um den Effekt sichtbar zu machen, schläft jedes Tool `LATENCY` Sekunden — wie ein echter
Netz- oder DB-Aufruf. Die Daten sind fest verdrahtet (kein Internet nötig).
Drei unabhängige Auskünfte pro Stadt: **Wetter**, **Einwohner**, **Zeitzone**.

In [2]:
LATENCY = 0.8  # Sekunden pro Tool-Aufruf (simulierte IO-Latenz)

CITY_DB = {
    "wien":   {"wetter": "18 Grad, bewoelkt", "einwohner": "2.0 Mio",  "zeitzone": "CET (UTC+1)"},
    "berlin": {"wetter": "15 Grad, Regen",    "einwohner": "3.7 Mio",  "zeitzone": "CET (UTC+1)"},
    "tokio":  {"wetter": "26 Grad, sonnig",   "einwohner": "14.0 Mio", "zeitzone": "JST (UTC+9)"},
    "wels":   {"wetter": "17 Grad, Nebel",    "einwohner": "0.06 Mio", "zeitzone": "CET (UTC+1)"},
}

def _lookup(city, feld):
    time.sleep(LATENCY)                       # <- simulierte Latenz
    return CITY_DB.get(city.strip().lower(), {}).get(feld, "unbekannt")

def wetter(stadt):    return _lookup(stadt, "wetter")
def einwohner(stadt): return _lookup(stadt, "einwohner")
def zeitzone(stadt):  return _lookup(stadt, "zeitzone")

CITY_TOOLS = [
    {"type": "function", "function": {"name": "wetter",
        "description": "Aktuelles Wetter einer Stadt.",
        "parameters": {"type": "object", "properties": {"stadt": {"type": "string"}}, "required": ["stadt"]}}},
    {"type": "function", "function": {"name": "einwohner",
        "description": "Einwohnerzahl einer Stadt.",
        "parameters": {"type": "object", "properties": {"stadt": {"type": "string"}}, "required": ["stadt"]}}},
    {"type": "function", "function": {"name": "zeitzone",
        "description": "Zeitzone einer Stadt.",
        "parameters": {"type": "object", "properties": {"stadt": {"type": "string"}}, "required": ["stadt"]}}},
]

CITY_DISPATCH = {"wetter": lambda a: wetter(a["stadt"]),
                 "einwohner": lambda a: einwohner(a["stadt"]),
                 "zeitzone": lambda a: zeitzone(a["stadt"])}

print("3 Tools definiert, je", LATENCY, "s Latenz. Staedte:", ", ".join(CITY_DB))

3 Tools definiert, je 0.8 s Latenz. Staedte: wien, berlin, tokio, wels


## 2 · Eine Antwort, mehrere `tool_calls`

Der Schlüssel: Bittet man das Modell um **unabhängige** Infos, packt es oft **mehrere**
Tool-Aufrufe in **eine** Assistant-Nachricht. Schauen wir uns das roh an — *ohne* Loop.

In [3]:
probe = [
    {"role": "system", "content": "Du beantwortest Fragen mit den Tools."},
    {"role": "user", "content": "Wie ist Wetter, Einwohnerzahl und Zeitzone von Wien?"},
]
msg = llm(probe, tools=CITY_TOOLS).choices[0].message
print("Anzahl tool_calls in EINER Antwort:", len(msg.tool_calls or []))
for tc in (msg.tool_calls or []):
    print("  •", tc.function.name, tc.function.arguments)

Anzahl tool_calls in EINER Antwort: 3
  • wetter {"stadt": "Wien"}
  • einwohner {"stadt": "Wien"}
  • zeitzone {"stadt": "Wien"}


Drei Aufrufe in einer Runde. Naiv führt man sie **nacheinander** aus → 3 × `LATENCY`.
Sie sind aber voneinander unabhängig → sie könnten **gleichzeitig** laufen → ≈ 1 × `LATENCY`.

## 3 · Ein Loop, zwei Ausführungsmodi

Derselbe Agent-Loop wie immer — mit **einem** Schalter `parallel`. Ist eine Tool-Batch
größer als 1 und `parallel=True`, schicken wir sie durch einen `ThreadPoolExecutor`,
sonst seriell. Beide Wege hängen für **jeden** `tool_call` ein `tool`-Result an (Pflicht,
sonst meckert die API).

In [12]:
def _pack(tc):
    return {"id": tc.id, "type": "function",
            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}

def run_agent(system, task, tools, dispatch, parallel=True, tag="agent",
              max_steps=12, verbose=True):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        msg = llm(messages, tools=tools).choices[0].message
        a = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            a["tool_calls"] = [_pack(tc) for tc in msg.tool_calls]
        messages.append(a)

        if not msg.tool_calls:
            return msg.content                       # fertig: finale Textantwort

        def run_one(tc):
            args = json.loads(tc.function.arguments or "{}")
            return {"role": "tool", "tool_call_id": tc.id,
                    "content": str(dispatch[tc.function.name](args))}

        names = ", ".join(tc.function.name for tc in msg.tool_calls)
        mode = "parallel" if (parallel and len(msg.tool_calls) > 1) else "seriell"
        if verbose:
            vprint(f"  [{tag}] Schritt {step}: {len(msg.tool_calls)} Tool(s) [{mode}] -> {names}")

        if parallel and len(msg.tool_calls) > 1:
            with ThreadPoolExecutor(max_workers=8) as ex:
                results = list(ex.map(run_one, msg.tool_calls))   # ex.map erhaelt die Reihenfolge
        else:
            results = [run_one(tc) for tc in msg.tool_calls]
        messages.extend(results)
    return "(max_steps erreicht)"

## 4 · Seriell vs. parallel — dieselbe Aufgabe, gestoppt

Wir fragen nach **drei** Auskünften und messen die Wanduhr. Inhaltlich identisch,
nur der Schalter `parallel` unterscheidet sich.

In [5]:
AUFGABE = "Nenne Wetter, Einwohnerzahl und Zeitzone von Wien. Antworte in einem Satz."

t0 = time.perf_counter()
out_seriell = run_agent("Du beantwortest knapp mit den Tools.", AUFGABE,
                        CITY_TOOLS, CITY_DISPATCH, parallel=False, tag="seriell")
t_seriell = time.perf_counter() - t0
print(f"SERIELL:  {t_seriell:5.1f}s  ->  {out_seriell}\n")

t0 = time.perf_counter()
out_parallel = run_agent("Du beantwortest knapp mit den Tools.", AUFGABE,
                         CITY_TOOLS, CITY_DISPATCH, parallel=True, tag="parallel")
t_parallel = time.perf_counter() - t0
print(f"PARALLEL: {t_parallel:5.1f}s  ->  {out_parallel}\n")

print(f"=> Speedup ca. {t_seriell / t_parallel:.1f}x  (3 Tools je {LATENCY}s)")

  [seriell] Schritt 1: 3 Tool(s) [seriell] -> wetter, einwohner, zeitzone
SERIELL:    4.2s  ->  Wien hat aktuell 18 Grad und ist bewölkt, hat etwa 2,0 Mio. Einwohner und liegt in der Zeitzone CET (UTC+1).

  [parallel] Schritt 1: 3 Tool(s) [parallel] -> wetter, einwohner, zeitzone
PARALLEL:   2.6s  ->  Wien hat aktuell 18 Grad und ist bewölkt, hat etwa 2,0 Mio. Einwohner und liegt in der Zeitzone CET (UTC+1).

=> Speedup ca. 1.6x  (3 Tools je 0.8s)


Seriell summieren sich die Latenzen (~3 × `LATENCY` plus LLM-Runden), parallel zahlt man
sie nur **einmal**. Genau das macht ein Agent-Harness im Hintergrund, wenn er „Tool-Calls
parallelisiert".

> ⚠️ Nur **nebenläufigkeitssichere** Tools parallel ausführen. Reine Lookups/Reads: ja.
> Schreibende Tools mit Seiteneffekten (gleiche Datei, gemeinsamer Zustand): besser seriell
> oder mit Lock — sonst Races.

## 5 · Sub-Agent als Tool — `delegate(auftrag)`

Jetzt der zweite Schritt: Ein Tool, das selbst ein **ganzer Agent** ist.

Der **Orchestrator** kennt nur **ein** Tool: `delegate`. Jeder Aufruf startet einen frischen
**Recherche-Sub-Agent** mit *eigenem* Kontext und den City-Tools, lässt ihn arbeiten und gibt
dessen **Ergebnis** als Tool-Result zurück. Der Orchestrator sieht nie die rohen Tool-Aufrufe
des Sub-Agents — nur das fertige Resultat. Das ist **Kontext-Isolation**: jeder Sub-Agent hat
seinen eigenen, sauberen Nachrichtenverlauf.

In [6]:
RECHERCHE_SYS = (
    "Du bist ein Recherche-Agent fuer GENAU EINE Stadt. "
    "Hole Wetter, Einwohner und Zeitzone mit deinen Tools (ruhig alle in einer Runde) "
    "und gib EINEN kompakten Steckbrief zurueck: 'Stadt: Wetter | Einwohner | Zeitzone'."
)

def delegate(auftrag):
    """Tool des Orchestrators: startet einen eigenstaendigen Sub-Agent-Loop."""
    tag = "sub:" + auftrag.strip()[:16]
    ergebnis = run_agent(RECHERCHE_SYS, auftrag, CITY_TOOLS, CITY_DISPATCH,
                         parallel=True, tag=tag, verbose=True)
    return ergebnis

ORCH_TOOLS = [
    {"type": "function", "function": {"name": "delegate",
        "description": ("Delegiert EINEN Rechercheauftrag an einen Sub-Agenten und gibt dessen "
                        "Ergebnis zurueck. Fuer mehrere Staedte: delegate mehrfach in DERSELBEN "
                        "Antwort aufrufen, damit die Sub-Agenten parallel laufen."),
        "parameters": {"type": "object",
                       "properties": {"auftrag": {"type": "string",
                           "description": "z. B. 'Steckbrief fuer Wien'"}},
                       "required": ["auftrag"]}}},
]
ORCH_DISPATCH = {"delegate": lambda a: delegate(a["auftrag"])}

ORCH_SYS = (
    "Du bist ein Koordinator. Zerlege die Aufgabe in unabhaengige Teilaufgaben und rufe fuer "
    "JEDE Stadt 'delegate' auf - alle in DERSELBEN Antwort (mehrere Tool-Calls gleichzeitig), "
    "damit die Sub-Agenten parallel arbeiten. Fasse die Steckbriefe danach als Markdown-Tabelle "
    "zusammen (Spalten: Stadt, Wetter, Einwohner, Zeitzone)."
)
print("Orchestrator + delegate-Tool bereit.")

Orchestrator + delegate-Tool bereit.


## 6 · Demo — Orchestrator delegiert drei Sub-Agents *parallel*

Beobachte die Spur: Der Orchestrator feuert **drei** `delegate`-Calls in einer Runde
(parallele Tool-Calls) → die drei Sub-Agenten laufen **gleichzeitig**, und jeder ruft
seine drei City-Tools wiederum **parallel** auf. Die `[sub:...]`-Zeilen verschiedener
Städte mischen sich — der sichtbare Beweis der Nebenläufigkeit.

In [11]:
t0 = time.perf_counter()
bericht = run_agent(ORCH_SYS,
                    "Vergleiche Wien, Berlin und Tokio.",
                    ORCH_TOOLS, ORCH_DISPATCH, parallel=True, tag="ORCH", verbose=True)
dauer = time.perf_counter() - t0

print(f"\n--- Orchestrator fertig in {dauer:.1f}s ---\n")
print(bericht)

  [ORCH] Schritt 1: 3 Tool(s) [parallel] -> delegate, delegate, delegate
  [sub:Steckbrief fuer ] Schritt 1: 3 Tool(s) [parallel] -> wetter, einwohner, zeitzone
  [sub:Steckbrief fuer ] Schritt 1: 3 Tool(s) [parallel] -> wetter, einwohner, zeitzone
  [sub:Steckbrief fuer ] Schritt 1: 3 Tool(s) [parallel] -> wetter, einwohner, zeitzone

--- Orchestrator fertig in 5.7s ---

| Stadt | Wetter | Einwohner | Zeitzone |
|---|---:|---:|---|
| Wien | 18 Grad, bewölkt | 2.0 Mio | CET (UTC+1) |
| Berlin | 15 Grad, Regen | 3.7 Mio | CET (UTC+1) |
| Tokio | 26 Grad, sonnig | 14.0 Mio | JST (UTC+9) |


Drei Sub-Agenten × drei Tools = **9** latenzbehaftete Aufrufe. Vollständig seriell wären das
~9 × `LATENCY`; durch die zwei Parallelisierungs-Ebenen (Sub-Agenten parallel, Tools je
Sub-Agent parallel) landet die reine Tool-Zeit bei ~1 × `LATENCY` — der Rest sind LLM-Runden.

## 7 · Interaktiv

Eigene Aufgabe an den Orchestrator. Bekannte Städte: **Wien, Berlin, Tokio, Wels**.
Leere Eingabe oder `exit` beendet.

In [ ]:
while True:
    frage = input("🧑 Aufgabe an den Orchestrator (oder 'exit'): ").strip()
    if not frage or frage.lower() in ("exit", "quit", "ende"):
        print("Beendet."); break
    print(run_agent(ORCH_SYS, frage, ORCH_TOOLS, ORCH_DISPATCH,
                    parallel=True, tag="ORCH", verbose=True))

## Mitnehmen

1. **Mehrere `tool_calls` pro Antwort sind die Norm**, nicht die Ausnahme. Ob du sie seriell
   oder parallel abarbeitest, ist eine reine **Harness-Entscheidung** — das Modell merkt davon nichts.
2. **Parallelität ist ein Dreizeiler.** `ThreadPoolExecutor` + jeder `tool_call` bekommt sein
   `tool`-Result. Für IO-lastige, unabhängige Tools bringt das den größten Hebel.
3. **Nur sichere Tools parallelisieren.** Reads/Lookups problemlos; schreibende Tools mit
   geteiltem Zustand brauchen Locks oder serielle Ausführung.
4. **Ein Sub-Agent ist nur ein Tool**, dessen Implementierung wieder `run_agent(...)` ruft.
   Das gibt **Kontext-Isolation** (eigener Nachrichtenverlauf) und **Spezialisierung**
   (eigener System-Prompt, eigene Tools).
5. **Die Muster stapeln sich.** Orchestrator delegiert Sub-Agenten *parallel*, jeder Sub-Agent
   nutzt seine Tools *parallel* — Supervisor/Map-Reduce, alles mit **demselben** Loop.

> Tools = **Fähigkeiten**, Skills = **Vorgehen**, Sub-Agenten = **Arbeitsteilung**.
> Drei Bausteine, ein und derselbe Agent-Loop.